# LeetCode 2089 — Find Target Indices After Sorting Array bằng Python

**Link đề:** https://leetcode.com/problems/find-target-indices-after-sorting-array/
**Độ khó:** Easy · **Chủ đề:** Array, Binary Search, Sorting

```python
class Solution:
    def targetIndices(self, nums: list[int], target: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu "chỉ số sau khi sắp xếp" và vì sao các chỉ số của `target` luôn **liên tiếp**.
2. Đi từ **sắp xếp + duyệt** (`O(n log n)`) → **sắp xếp + binary search** (`bisect`) → **đếm không cần sắp xếp** (`O(n)`).
3. Nắm cú pháp: `sorted` vs `.sort()`, `enumerate`, list comprehension, `range(a, b)`, `bisect_left/right`.
4. Trace từng bước, chứng minh, phân tích độ phức tạp, so sánh các cách.
5. Luyện biến thể: tìm khoảng `[first, last]` của target (LeetCode 34), đếm số phần tử nhỏ hơn.

## 1. Đề bài và cách hiểu đúng

Cho mảng `nums` (chỉ số từ 0) và số `target`. **Sắp xếp `nums` tăng dần (không giảm)**, rồi trả về **danh sách các chỉ số `i`** sao cho `nums[i] == target` (danh sách tăng dần). Không có thì trả `[]`.

| `nums` | `target` | Sau sắp xếp | Kết quả |
|---|:---:|---|---|
| `[1,2,5,2,3]` | 2 | `[1,2,2,3,5]` | `[1,2]` |
| `[1,2,5,2,3]` | 3 | `[1,2,2,3,5]` | `[3]` |
| `[1,2,5,2,3]` | 5 | `[1,2,2,3,5]` | `[4]` |
| `[1,2,5,2,3]` | 4 | `[1,2,2,3,5]` | `[]` |

**Constraints:** `1 <= len(nums) <= 100`, `1 <= nums[i], target <= 100`.

Chú ý: chỉ số là chỉ số **trong mảng đã sắp xếp**, không phải trong mảng gốc. Trong mảng gốc, số 2 ở vị trí 1 và 3, nhưng đáp án là `[1, 2]`.

In [1]:
examples = [
    ([1, 2, 5, 2, 3], 2, [1, 2]),
    ([1, 2, 5, 2, 3], 3, [3]),
    ([1, 2, 5, 2, 3], 5, [4]),
]
nums = [1, 2, 5, 2, 3]
print("Chỉ số của 2 trong mảng GỐC:", [i for i in range(len(nums)) if nums[i] == 2])
print("Mảng sau sắp xếp:", sorted(nums))

Chỉ số của 2 trong mảng GỐC: [1, 3]
Mảng sau sắp xếp: [1, 2, 2, 3, 5]


## 2. Nền tảng Python cần dùng

### 2.1. `sorted(nums)` và `nums.sort()`

- `sorted(nums)` trả về **list mới** đã sắp xếp, không đổi `nums`.
- `nums.sort()` sắp xếp **tại chỗ** và trả về `None`.

Cả hai đều `O(n log n)` (thuật toán Timsort).

In [2]:
a = [3, 1, 2]
b2 = sorted(a)
print("a =", a, "| sorted(a) =", b2)
r = a.sort()
print("sau a.sort(): a =", a, "| giá trị trả về =", r)

a = [3, 1, 2] | sorted(a) = [1, 2, 3]
sau a.sort(): a = [1, 2, 3] | giá trị trả về = None


### 2.2. `enumerate` — vừa lấy chỉ số vừa lấy giá trị

`for i, x in enumerate(arr):` cho `i = 0, 1, 2, ...` và `x = arr[i]`. Gọn hơn `for i in range(len(arr))`.

In [3]:
for i, x in enumerate([1, 2, 2, 3, 5]):
    print(i, x)

0 1
1 2
2 2
3 3
4 5


### 2.3. List comprehension

`[biểu_thức for phần_tử in dãy if điều_kiện]` tạo list mới trong một dòng.

In [4]:
arr = [1, 2, 2, 3, 5]
print([i for i, x in enumerate(arr) if x == 2])   # [1, 2]
print([x * 10 for x in arr if x > 2])             # [30, 50]

[1, 2]
[30, 50]


### 2.4. `range(a, b)` và `list(range(...))`

`range(a, b)` sinh `a, a+1, ..., b-1` (không gồm `b`). Nếu `a == b` thì rỗng. Đây là chìa khóa của lời giải `O(n)`.

In [5]:
print(list(range(1, 3)))   # [1, 2]
print(list(range(3, 4)))   # [3]
print(list(range(4, 4)))   # []  -> rỗng

[1, 2]
[3]
[]


### 2.5. Module `bisect` — binary search trên list đã sắp xếp

- `bisect_left(arr, x)`: vị trí **đầu tiên** có thể chèn `x` mà vẫn giữ thứ tự = số phần tử `< x`.
- `bisect_right(arr, x)`: vị trí **sau phần tử `x` cuối cùng** = số phần tử `<= x`.

Mọi vị trí `i` với `bisect_left <= i < bisect_right` đều có `arr[i] == x`.

In [6]:
from bisect import bisect_left, bisect_right
arr = [1, 2, 2, 3, 5]
for x in [2, 3, 4, 0, 6]:
    print(f"x={x}: bisect_left={bisect_left(arr, x)}, bisect_right={bisect_right(arr, x)}")

x=2: bisect_left=1, bisect_right=3
x=3: bisect_left=3, bisect_right=4
x=4: bisect_left=4, bisect_right=4
x=0: bisect_left=0, bisect_right=0
x=6: bisect_left=5, bisect_right=5


## 3. Cách 1 — Trực giác: làm đúng như đề nói

Gợi ý của đề: *"Try sorting the array first. Now find all indices whose values are equal to target."*

1. Sắp xếp.
2. Duyệt từng chỉ số, giữ các `i` có `arr[i] == target`.

In [7]:
def target_indices_sort_scan(nums: list[int], target: int) -> list[int]:
    arr = sorted(nums)
    result = []
    for i, x in enumerate(arr):
        if x == target:
            result.append(i)
    return result

for nums, t, exp in examples:
    print(target_indices_sort_scan(nums, t), "kỳ vọng", exp)

[1, 2] kỳ vọng [1, 2]
[3] kỳ vọng [3]
[4] kỳ vọng [4]


## 4. Nút thắt — có thật sự cần sắp xếp không?

Sắp xếp tốn `O(n log n)`. Với `n ≤ 100` thì chẳng sao, nhưng câu hỏi phỏng vấn thường là: **làm được `O(n)` không?**

Quan sát: trong mảng đã sắp xếp, các phần tử bằng `target` **nằm thành một khối liên tiếp**:

```text
[ (các số < target) | (các số == target) | (các số > target) ]
   ^ less phần tử       ^ equal phần tử
```

Khối đó bắt đầu ở chỉ số `less` (số phần tử nhỏ hơn target) và có độ dài `equal`. Vậy đáp án là:

```python
list(range(less, less + equal))
```

Hai con số này đếm được **mà không cần sắp xếp**!

In [8]:
nums, target = [1, 2, 5, 2, 3], 2
less = sum(1 for x in nums if x < target)
equal = sum(1 for x in nums if x == target)
print("less =", less, "| equal =", equal, "| đáp án =", list(range(less, less + equal)))

less = 1 | equal = 2 | đáp án = [1, 2]


## 5. Bước đột phá — chỉ số đầu tiên = số phần tử nhỏ hơn

Tại sao chỉ số đầu của khối `target` bằng `less`?

- Sắp xếp tăng dần đặt **mọi** phần tử `< target` lên trước phần tử bằng `target`.
- Có đúng `less` phần tử như vậy, chiếm chỉ số `0 .. less-1`.
- Nên phần tử `target` đầu tiên nằm ở chỉ số `less`, các phần tử tiếp theo nằm ở `less+1, ...`, tổng cộng `equal` phần tử.

Ví dụ `[1,2,5,2,3]`, target = 3: `less = 3` (1, 2, 2), `equal = 1` → `[3]`. Đúng với ví dụ 2.

In [9]:
for nums, t, exp in examples:
    less = sum(1 for x in nums if x < t)
    equal = nums.count(t)
    print(f"target={t}: less={less}, equal={equal} -> {list(range(less, less + equal))} (kỳ vọng {exp})")

target=2: less=1, equal=2 -> [1, 2] (kỳ vọng [1, 2])
target=3: less=3, equal=1 -> [3] (kỳ vọng [3])
target=5: less=4, equal=1 -> [4] (kỳ vọng [4])


## 6. Lời giải chính (đếm, `O(n)`) — mổ xẻ rồi ghép

### 6.1. Đoạn 1: một vòng lặp đếm cả hai đại lượng

Thay vì hai lần `sum(...)`, ta đếm trong **một** vòng for với `if / elif`.

In [10]:
nums, target = [1, 2, 5, 2, 3], 2
less = equal = 0          # gán cùng lúc hai biến bằng 0
for x in nums:
    if x < target:
        less += 1
    elif x == target:
        equal += 1
    # x > target: bỏ qua
print(less, equal)

1 2


### 6.2. Đoạn 2: dựng kết quả từ `range`

`list(range(less, less + equal))`. Nếu `equal == 0` thì ra `[]` tự động — không cần `if` riêng.

In [11]:
print(list(range(3, 3 + 0)))   # target không có -> []
print(list(range(1, 1 + 2)))   # [1, 2]

[]
[1, 2]


### 6.3. Ghép lại

In [12]:
def target_indices_count(nums: list[int], target: int) -> list[int]:
    less = equal = 0
    for x in nums:
        if x < target:
            less += 1
        elif x == target:
            equal += 1
    return list(range(less, less + equal))

for nums, t, exp in examples + [([1, 2, 5, 2, 3], 4, [])]:
    print(target_indices_count(nums, t), "kỳ vọng", exp)

[1, 2] kỳ vọng [1, 2]
[3] kỳ vọng [3]
[4] kỳ vọng [4]
[] kỳ vọng []


## 7. TRACE từng bước

### 7.1. Bảng trace — `nums = [1, 2, 5, 2, 3]`, `target = 2`

| Bước | x | So với target | less | equal |
|:---:|:---:|---|:---:|:---:|
| 1 | 1 | `<` → less += 1 | 1 | 0 |
| 2 | 2 | `==` → equal += 1 | 1 | 1 |
| 3 | 5 | `>` → bỏ qua | 1 | 1 |
| 4 | 2 | `==` → equal += 1 | 1 | 2 |
| 5 | 3 | `>` → bỏ qua | 1 | 2 |

Kết thúc: `range(1, 3)` → `[1, 2]`.

In [13]:
def trace_target_indices(nums, target):
    less = equal = 0
    print(f"nums={nums}, target={target}")
    for step, x in enumerate(nums, start=1):
        if x < target:
            less += 1; action = "<  -> less += 1"
        elif x == target:
            equal += 1; action = "== -> equal += 1"
        else:
            action = ">  -> bỏ qua"
        print(f"  bước {step}: x={x}  {action:18} less={less}, equal={equal}")
    res = list(range(less, less + equal))
    print(f"  => range({less}, {less + equal}) = {res}")
    return res

trace_target_indices([1, 2, 5, 2, 3], 2)

nums=[1, 2, 5, 2, 3], target=2
  bước 1: x=1  <  -> less += 1    less=1, equal=0
  bước 2: x=2  == -> equal += 1   less=1, equal=1
  bước 3: x=5  >  -> bỏ qua       less=1, equal=1
  bước 4: x=2  == -> equal += 1   less=1, equal=2
  bước 5: x=3  >  -> bỏ qua       less=1, equal=2
  => range(1, 3) = [1, 2]


[1, 2]

### 7.2. Nhánh đặc biệt: target không xuất hiện, và target nhỏ nhất / lớn nhất

In [14]:
trace_target_indices([1, 2, 5, 2, 3], 4)   # không có -> []
print()
trace_target_indices([7, 7, 7], 7)         # toàn bộ bằng target
print()
trace_target_indices([9, 8, 1], 1)         # target nhỏ nhất -> chỉ số 0

nums=[1, 2, 5, 2, 3], target=4
  bước 1: x=1  <  -> less += 1    less=1, equal=0
  bước 2: x=2  <  -> less += 1    less=2, equal=0
  bước 3: x=5  >  -> bỏ qua       less=2, equal=0
  bước 4: x=2  <  -> less += 1    less=3, equal=0
  bước 5: x=3  <  -> less += 1    less=4, equal=0
  => range(4, 4) = []

nums=[7, 7, 7], target=7
  bước 1: x=7  == -> equal += 1   less=0, equal=1
  bước 2: x=7  == -> equal += 1   less=0, equal=2
  bước 3: x=7  == -> equal += 1   less=0, equal=3
  => range(0, 3) = [0, 1, 2]

nums=[9, 8, 1], target=1
  bước 1: x=9  >  -> bỏ qua       less=0, equal=0
  bước 2: x=8  >  -> bỏ qua       less=0, equal=0
  bước 3: x=1  == -> equal += 1   less=0, equal=1
  => range(0, 1) = [0]


[0]

### 7.3. Trace cách binary search để so sánh

Với mảng đã sắp xếp `[1,2,2,3,5]`, `bisect_left(arr, 2) = 1` và `bisect_right(arr, 2) = 3` → `range(1, 3)`. Chú ý `bisect_left` chính là `less`, và `bisect_right - bisect_left` chính là `equal`.

In [15]:
arr = sorted([1, 2, 5, 2, 3])
for t in [2, 3, 5, 4]:
    lo, hi = bisect_left(arr, t), bisect_right(arr, t)
    print(f"target={t}: lo={lo}, hi={hi} -> {list(range(lo, hi))}")

target=2: lo=1, hi=3 -> [1, 2]
target=3: lo=3, hi=4 -> [3]
target=5: lo=4, hi=5 -> [4]
target=4: lo=4, hi=4 -> []


### 7.4. Tự viết binary search tìm vị trí đầu tiên `>= target`

Để hiểu `bisect_left` bên trong làm gì: giữ khoảng `[lo, hi)`; nếu `arr[mid] < target` thì đáp án ở bên phải, ngược lại ở bên trái (gồm `mid`).

In [16]:
def lower_bound(arr, target):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid
        print(f"   lo={lo}, hi={hi}")
    return lo

arr = [1, 2, 2, 3, 5]
print("lower_bound(arr, 2) =", lower_bound(arr, 2))
print("lower_bound(arr, 3) =", lower_bound(arr, 3))

   lo=0, hi=2
   lo=0, hi=1
   lo=1, hi=1
lower_bound(arr, 2) = 1
   lo=3, hi=5
   lo=3, hi=4
   lo=3, hi=3
lower_bound(arr, 3) = 3


## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — Sắp xếp + duyệt

In [17]:
class Solution:
    def targetIndices(self, nums: list[int], target: int) -> list[int]:
        nums = sorted(nums)
        return [i for i, x in enumerate(nums) if x == target]

print([Solution().targetIndices(n, t) for n, t, _ in examples])

[[1, 2], [3], [4]]


### Cách B — Sắp xếp + binary search (`bisect`)

In [18]:
from bisect import bisect_left, bisect_right

class Solution:
    def targetIndices(self, nums: list[int], target: int) -> list[int]:
        nums = sorted(nums)
        return list(range(bisect_left(nums, target), bisect_right(nums, target)))

print([Solution().targetIndices(n, t) for n, t, _ in examples])

[[1, 2], [3], [4]]


### Cách C — Đếm `less` và `equal` (tối ưu `O(n)`, lời giải chính)

In [19]:
class Solution:
    def targetIndices(self, nums: list[int], target: int) -> list[int]:
        less = equal = 0
        for x in nums:
            if x < target:
                less += 1
            elif x == target:
                equal += 1
        return list(range(less, less + equal))

print([Solution().targetIndices(n, t) for n, t, _ in examples])

[[1, 2], [3], [4]]


### Cách D — Counting sort (tận dụng giá trị nhỏ `1..100`)

Vì giá trị chỉ từ 1 đến 100, ta có thể đếm số lần của từng giá trị vào mảng `cnt[0..100]`. Khi đó `less = cnt[1] + ... + cnt[target-1]`. Cách này minh họa ý tưởng **counting sort** — rất hữu ích khi miền giá trị nhỏ.

In [20]:
class Solution:
    def targetIndices(self, nums: list[int], target: int) -> list[int]:
        cnt = [0] * 101
        for x in nums:
            cnt[x] += 1
        less = sum(cnt[:target])      # tổng các giá trị 0..target-1
        return list(range(less, less + cnt[target]))

print([Solution().targetIndices(n, t) for n, t, _ in examples])

[[1, 2], [3], [4]]


## 9. Vì sao thuật toán đúng?

**Mệnh đề:** trong mảng đã sắp xếp tăng dần `arr`, tập chỉ số `{i : arr[i] == target}` đúng bằng `{less, less+1, ..., less+equal-1}`.

**Chứng minh ngắn:**
1. Sắp xếp không thêm/bớt phần tử, nên số phần tử `< target` vẫn là `less`, số phần tử `== target` vẫn là `equal`.
2. Do tăng dần, mọi phần tử `< target` đứng trước mọi phần tử `== target`, và mọi phần tử `> target` đứng sau. Vậy `less` phần tử nhỏ hơn chiếm đúng các chỉ số `0..less-1`.
3. Tiếp theo là `equal` phần tử bằng target, chiếm `less..less+equal-1`.

**Invariant vòng lặp (Cách C):** sau khi xử lý `k` phần tử đầu, `less` = số phần tử `< target` trong `nums[:k]`, `equal` = số phần tử `== target` trong `nums[:k]`.

Với Cách B: `bisect_left` trả đúng `less`, `bisect_right` trả `less + equal`, nên hai cách cho cùng kết quả.

## 10. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V = 100` là giá trị lớn nhất.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| A. Sort + duyệt | `O(n log n)` | `O(n)` | dễ nghĩ nhất |
| B. Sort + bisect | `O(n log n)` (sort chiếm ưu thế; tìm chỉ `O(log n)`) | `O(n)` | hữu ích nếu mảng **đã** sắp xếp sẵn |
| C. Đếm less/equal | `O(n)` | `O(1)` (không tính output) | tối ưu |
| D. Counting sort | `O(n + V)` | `O(V)` | khi miền giá trị nhỏ |

Output có `equal` phần tử nên mọi cách đều tốn thêm `O(equal)` để tạo kết quả.

## 11. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [21]:
import random
from bisect import bisect_left, bisect_right

def sol_A(nums, t): return [i for i, x in enumerate(sorted(nums)) if x == t]
def sol_B(nums, t):
    a = sorted(nums); return list(range(bisect_left(a, t), bisect_right(a, t)))
def sol_C(nums, t): return target_indices_count(nums, t)
def sol_D(nums, t):
    cnt = [0] * 101
    for x in nums: cnt[x] += 1
    less = sum(cnt[:t]); return list(range(less, less + cnt[t]))

SOLS = [sol_A, sol_B, sol_C, sol_D]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:28} PASS {actual}")

tests = examples + [
    ([5], 5, [0]),                 # một phần tử, có target
    ([5], 4, []),                  # một phần tử, không có
    ([1, 1, 1, 1], 1, [0, 1, 2, 3]),
    ([100, 1, 50], 100, [2]),      # target lớn nhất
    ([100, 1, 50], 1, [0]),        # target nhỏ nhất
    ([3, 3, 1, 1], 2, []),         # target nằm "giữa" nhưng không có
]
for i, (n, t, exp) in enumerate(tests):
    for f in SOLS:
        check(f"test {i} {f.__name__}", f(n, t), exp)

test 0 sol_A                 PASS [1, 2]
test 0 sol_B                 PASS [1, 2]
test 0 sol_C                 PASS [1, 2]
test 0 sol_D                 PASS [1, 2]
test 1 sol_A                 PASS [3]
test 1 sol_B                 PASS [3]
test 1 sol_C                 PASS [3]
test 1 sol_D                 PASS [3]
test 2 sol_A                 PASS [4]
test 2 sol_B                 PASS [4]
test 2 sol_C                 PASS [4]
test 2 sol_D                 PASS [4]
test 3 sol_A                 PASS [0]
test 3 sol_B                 PASS [0]
test 3 sol_C                 PASS [0]
test 3 sol_D                 PASS [0]
test 4 sol_A                 PASS []
test 4 sol_B                 PASS []
test 4 sol_C                 PASS []
test 4 sol_D                 PASS []
test 5 sol_A                 PASS [0, 1, 2, 3]
test 5 sol_B                 PASS [0, 1, 2, 3]
test 5 sol_C                 PASS [0, 1, 2, 3]
test 5 sol_D                 PASS [0, 1, 2, 3]
test 6 sol_A                 PASS [2]
test 6

In [22]:
rng = random.Random(2089)
for _ in range(500):
    nums = [rng.randint(1, 10) for _ in range(rng.randint(1, 15))]
    t = rng.randint(1, 10)
    outs = [f(nums, t) for f in SOLS]
    assert all(o == outs[0] for o in outs), (nums, t, outs)
print("500 test ngẫu nhiên: 4 cách khớp nhau")

500 test ngẫu nhiên: 4 cách khớp nhau


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: Trả chỉ số trong mảng gốc (quên sắp xếp)

In [23]:
def wrong_no_sort(nums, target):
    return [i for i, x in enumerate(nums) if x == target]   # SAI

print("Sai :", wrong_no_sort([1, 2, 5, 2, 3], 2), "| Đúng:", sol_C([1, 2, 5, 2, 3], 2))

Sai : [1, 3] | Đúng: [1, 2]


**Sửa:** dùng `sorted(nums)` trước, hoặc dùng công thức đếm `less`.

### Lỗi 2: `nums = nums.sort()`

In [24]:
nums = [3, 1, 2]
nums = nums.sort()     # SAI: .sort() trả None
print("nums =", nums)
# Sửa: nums.sort()  (không gán)   hoặc   nums = sorted(nums)

nums = None


### Lỗi 3: Đếm `<=` thay vì `<` cho `less`

Nếu đếm `x <= target` thì `less` đã bao gồm cả các phần tử bằng target → khối bị lệch sang phải.

In [25]:
def wrong_le(nums, target):
    less = sum(1 for x in nums if x <= target)    # SAI
    equal = nums.count(target)
    return list(range(less, less + equal))

print("Sai :", wrong_le([1, 2, 5, 2, 3], 2), "| Đúng:", sol_C([1, 2, 5, 2, 3], 2))

Sai : [3, 4] | Đúng: [1, 2]


### Lỗi 4: Dùng `range(less, less + equal + 1)` (lệch một)

`range` không gồm đầu mút phải. Cộng thêm 1 sẽ trả dư một chỉ số (và trả `[less]` ngay cả khi target không có!).

In [26]:
less, equal = 3, 0
print("Sai :", list(range(less, less + equal + 1)), "| Đúng:", list(range(less, less + equal)))

Sai : [3] | Đúng: []


## 13. Mẹo nhận dạng pattern

- "Vị trí của x **sau khi sắp xếp**" = **số phần tử nhỏ hơn x** (rank). Không cần sắp xếp thật.
- Mảng đã sắp xếp + hỏi về khối giá trị bằng nhau → `bisect_left` / `bisect_right`.
- Miền giá trị nhỏ (≤ 10^5) → nghĩ tới **counting sort / mảng đếm**.

Bài liên quan: 34 Find First and Last Position of Element in Sorted Array, 1365 How Many Numbers Are Smaller Than the Current Number, 35 Search Insert Position.

## 14. Biến thể

### 14.1. Trả về `[first, last]` của target (kiểu LeetCode 34), `[-1, -1]` nếu không có

In [27]:
def first_last(nums, target):
    less = sum(1 for x in nums if x < target)
    equal = nums.count(target)
    return [less, less + equal - 1] if equal else [-1, -1]

print(first_last([1, 2, 5, 2, 3], 2), first_last([1, 2, 5, 2, 3], 4))

[1, 2] [-1, -1]


### 14.2. Với mỗi phần tử, đếm số phần tử nhỏ hơn nó (LeetCode 1365)

Dùng mảng đếm + tổng tiền tố: `prefix[v]` = số phần tử `< v`.

In [28]:
def smaller_numbers(nums):
    cnt = [0] * 102
    for x in nums:
        cnt[x + 1] += 1
    for v in range(1, 102):
        cnt[v] += cnt[v - 1]       # cnt[v] = số phần tử < v
    return [cnt[x] for x in nums]

print(smaller_numbers([8, 1, 2, 2, 3]))   # [4, 0, 1, 1, 3]

[4, 0, 1, 1, 3]


### 14.3. Sắp xếp giảm dần thì sao?

Khối target bắt đầu ở chỉ số = **số phần tử lớn hơn target**.

In [29]:
def target_indices_desc(nums, target):
    greater = sum(1 for x in nums if x > target)
    equal = nums.count(target)
    return list(range(greater, greater + equal))

print(target_indices_desc([1, 2, 5, 2, 3], 2), "| kiểm tra:", [i for i, x in enumerate(sorted([1,2,5,2,3], reverse=True)) if x == 2])

[2, 3] | kiểm tra: [2, 3]


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Các test để dạng comment. Đáp án ở **mục 17**.

### Bài 1 — Đếm số phần tử nhỏ hơn `t`

In [30]:
def count_less(nums, t):
    # TODO: viết code ở đây
    pass

# assert count_less([1, 2, 5, 2, 3], 3) == 3

### Bài 2 — Tự viết lại lời giải #2089 bằng cách đếm

In [31]:
def target_indices_ex(nums, target):
    # TODO: viết code ở đây
    pass

# assert target_indices_ex([1, 2, 5, 2, 3], 2) == [1, 2]
# assert target_indices_ex([1, 2, 5, 2, 3], 4) == []

### Bài 3 — Tự viết `upper_bound(arr, t)` (giống `bisect_right`) bằng binary search

In [32]:
def upper_bound(arr, t):
    # TODO: viết code ở đây
    pass

# assert upper_bound([1, 2, 2, 3, 5], 2) == 3
# assert upper_bound([1, 2, 2, 3, 5], 0) == 0

### Bài 4 — Biến thể: chỉ số của mọi phần tử nằm trong đoạn `[lo, hi]` sau sắp xếp
Ví dụ `nums=[1,2,5,2,3]`, `lo=2`, `hi=3` → sau sắp xếp `[1,2,2,3,5]` → `[1,2,3]`.

In [33]:
def range_indices(nums, lo, hi):
    # TODO: viết code ở đây
    pass

# assert range_indices([1, 2, 5, 2, 3], 2, 3) == [1, 2, 3]

### Bài 5 — Tự tạo test làm lộ **Lỗi 3** (`<=`)
Viết 3 bộ `(nums, target)` mà `wrong_le` cho kết quả khác `sol_C`. Có test nào `wrong_le` vô tình đúng không?

In [34]:
my_tests = [
    # TODO: (nums, target)
]
# for n, t in my_tests:
#     assert wrong_le(n, t) != sol_C(n, t)

### Bài 6 — Mini challenge: rank của mỗi phần tử (giá trị bằng nhau cùng rank, rank bắt đầu từ 1)
Ví dụ `[40, 10, 20, 10]` → `[3, 1, 2, 1]` (LeetCode 1331).

In [35]:
def array_rank(arr):
    # TODO: viết code ở đây
    pass

# assert array_rank([40, 10, 20, 10]) == [3, 1, 2, 1]

## 16. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Duyệt và đếm | `sum(1 for ...)` | `sum(1 for x in nums if x < t)` |
| 2 | Khối liên tiếp | đếm less & equal | `list(range(less, less + equal))` |
| 3 | Giống lower_bound | đổi điều kiện | `if arr[mid] <= t: lo = mid + 1` |
| 4 | Khối `[lo, hi]` cũng liên tiếp | bắt đầu = số phần tử `< lo` | độ dài = số phần tử trong `[lo, hi]` |
| 5 | `wrong_le` dịch khối sang phải | sai khi target có mặt | khi target vắng, `<=` và `<` cho cùng `less` |
| 6 | Rank = số giá trị **khác nhau** nhỏ hơn + 1 | `sorted(set(arr))` | dict `value -> rank` |

## 17. Đáp án tham khảo

In [36]:
def count_less(nums, t):
    return sum(1 for x in nums if x < t)

def target_indices_ex(nums, target):
    less = count_less(nums, target)
    equal = nums.count(target)
    return list(range(less, less + equal))

def upper_bound(arr, t):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] <= t:
            lo = mid + 1
        else:
            hi = mid
    return lo

def range_indices(nums, lo, hi):
    start = sum(1 for x in nums if x < lo)
    size = sum(1 for x in nums if lo <= x <= hi)
    return list(range(start, start + size))

my_tests = [([1, 2, 5, 2, 3], 2), ([7, 7], 7), ([1, 3], 1)]

def array_rank(arr):
    rank = {v: i + 1 for i, v in enumerate(sorted(set(arr)))}
    return [rank[v] for v in arr]

assert count_less([1, 2, 5, 2, 3], 3) == 3
assert target_indices_ex([1, 2, 5, 2, 3], 2) == [1, 2]
assert target_indices_ex([1, 2, 5, 2, 3], 4) == []
assert upper_bound([1, 2, 2, 3, 5], 2) == 3 and upper_bound([1, 2, 2, 3, 5], 0) == 0
assert range_indices([1, 2, 5, 2, 3], 2, 3) == [1, 2, 3]
for n, t in my_tests:
    assert wrong_le(n, t) != sol_C(n, t)
print("wrong_le vô tình đúng khi target vắng mặt:", wrong_le([1, 3], 2) == sol_C([1, 3], 2))
assert array_rank([40, 10, 20, 10]) == [3, 1, 2, 1]
print("Tất cả đáp án PASS")

wrong_le vô tình đúng khi target vắng mặt: True
Tất cả đáp án PASS


**Giải thích bằng lời:**
- Bài 1–2: công thức `range(less, less + equal)`.
- Bài 3: `upper_bound` khác `lower_bound` duy nhất ở `<=` — nó bỏ qua cả các phần tử bằng `t`.
- Bài 4: khối giá trị trong `[lo, hi]` cũng liên tiếp sau sắp xếp, bắt đầu tại số phần tử `< lo`.
- Bài 5: khi target vắng thì `equal = 0` nên kết quả luôn `[]` → lỗi bị che. Test phải chứa target.
- Bài 6: rank dùng set để gộp giá trị trùng rồi đánh số theo thứ tự tăng dần.

## 18. Tổng kết — template + checklist

**Template:** "chỉ số sau khi sắp xếp" ⇒ đếm `less` (số phần tử nhỏ hơn) và `equal`; đáp án là `range(less, less + equal)`.

```python
less = sum(x < target for x in nums)
equal = nums.count(target)
return list(range(less, less + equal))
```

**Checklist trước khi submit:**
- [ ] Chỉ số là của mảng **đã sắp xếp**.
- [ ] Đếm `<` (không phải `<=`) cho `less`.
- [ ] `range` không cộng thêm 1.
- [ ] Không gán `nums = nums.sort()`.
- [ ] Target vắng mặt trả `[]`.
- [ ] Chạy lại 3 ví dụ: `[1,2]`, `[3]`, `[4]`.